# Lab 79: The Imitation Game

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 9, *The Future Interface*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-79.ipynb)

**What you will do:** chat blind with a small open chatbot running in your own Colab notebook, log how confident you were that you were talking to a machine, and compare your judgement accuracy with the controlled study the book reports.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 79 you held a conversation with an AI chatbot and with a friend, then tried to spot which was
which. The book reports a controlled version of this test: Jones and Bergen (2024) found that GPT-4 was
judged human in about half of its conversations -- a genuine pass by Turing's own criterion -- while real
human witnesses were judged human about two-thirds of the time, and the much simpler ELIZA program fooled
judges only about a fifth of the time.

## Record your results

Log each conversation you tried to judge: who or what you were actually talking to (fill this in
honestly only after you've made your guess), your guess, your confidence, and the main cue you used to
decide.

In [ ]:
import pandas as pd

# example data: replace with your own conversations and guesses
trials = pd.DataFrame([
    {"trial": 1, "actually": "human",   "your_guess": "human",   "confidence_pct": 70, "cue": "personal anecdote"},
    {"trial": 2, "actually": "chatbot", "your_guess": "chatbot", "confidence_pct": 60, "cue": "overly balanced answer"},
    {"trial": 3, "actually": "chatbot", "your_guess": "human",   "confidence_pct": 55, "cue": "made a typo"},
    {"trial": 4, "actually": "human",   "your_guess": "human",   "confidence_pct": 80, "cue": "in-joke reference"},
])  # example data: replace with your own
trials["correct"] = trials["actually"] == trials["your_guess"]
display(trials)
print(f"Your accuracy: {trials['correct'].mean():.0%} across {len(trials)} trials")

## Compare

The book reports specific accuracy figures from the controlled study: GPT-4 was judged human in about
half of conversations (a coin-flip result -- chance performance at telling human from AI), human
witnesses were judged human about two-thirds of the time, and ELIZA fooled judges about a fifth of the
time.

In [ ]:
your_accuracy = trials["correct"].mean()
print(f"Your accuracy at telling human from chatbot: {your_accuracy:.0%}")
print("Benchmarks from the Lab's own text: a human witness is judged human about 67% of the time; GPT-4")
print("is judged human about 50% of the time (chance); old-fashioned ELIZA is judged human about 20% of")
print("the time.")
if your_accuracy >= 0.8:
    print("You did notably better than the interrogators in the published study -- with only a handful of")
    print("trials, and a chatbot you may already suspect, that is not surprising.")
else:
    print("Your accuracy is roughly in the range the published study found for judging real AI witnesses --")
    print("consistent with the book's claim that this is a genuinely hard discrimination, not an easy one.")

## The AI side

This Lab is naturally AI-native: chatting with a free, open chat model *is* the AI Lab. Below, a small
open instruct model (about 500 million parameters -- tiny compared with GPT-4, but enough to hold a
fluent conversation) answers a few questions. Read its answers without knowing in advance which
questions might trip it up, then judge them the way you judged your Step 1-3 conversations above.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
!pip install -q transformers accelerate

from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto")

prompts = [
    "What's the most interesting thing you've read about recently?",
    "Have you ever made a decision you regretted? What happened?",
    "What's 47 times 36? Just the number, please.",
    "Describe the smell of rain on hot pavement.",
]

for prompt in prompts:
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt")
    output_ids = model.generate(**inputs, max_new_tokens=80, do_sample=True, temperature=0.7)
    reply = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"Q: {prompt}\nA: {reply.strip()}\n")

Read back over the model's answers. Small open models like this one are far behind GPT-4: they are more
likely to get simple arithmetic wrong, to answer personal questions with generic or inconsistent content
(this model has no life to have made decisions in, so any "regret" it describes is invented), and to
lose track of context in a longer exchange. If any answer sounded convincingly personal, that is your
social-prediction network doing what the book describes -- attributing a mind to fluent language -- not
evidence that the model has experiences to report. A model this size is also far more likely than GPT-4
to produce confident nonsense, so treat anything factual it says with real scepticism.

## Pool the class data

Pool judgement accuracy across the group, together with the main cue each person used, using an
anonymous ID (never a name). This lets you ask a sharper question than any one person's data can answer:
across many judges, is accuracy at telling human from AI actually better than chance?

In [ ]:
import io

csv_text = """id,trial,actually,guess,cue
P01,1,human,human,personal anecdote
P01,2,chatbot,chatbot,overly balanced answer
P02,1,chatbot,human,fluent grammar
P02,2,human,human,typo
P03,1,chatbot,chatbot,generic phrasing
P03,2,human,chatbot,slow reply
P04,1,human,human,in-joke reference
P04,2,chatbot,chatbot,repetitive structure
"""  # example data: replace with your class CSV
class_df = pd.read_csv(io.StringIO(csv_text))
class_df["correct"] = class_df["actually"] == class_df["guess"]

boot = [class_df["correct"].sample(len(class_df), replace=True).mean() for _ in range(5000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Class accuracy: {class_df['correct'].mean():.0%} "
      f"(95% bootstrap interval {lo:.0%}-{hi:.0%}, n = {len(class_df)} judgements)")
display(class_df["cue"].value_counts().rename("times used as the deciding cue"))

A bootstrap interval that comfortably excludes 50% suggests the group really can do better than a coin
flip; an interval straddling 50% is consistent with the book's report that GPT-4-calibre models can push
human judges down towards chance. With only a handful of pooled trials, treat this as a demonstration of
the method, not a conclusive test.

## Questions to think about

1. Look at the cues you and your classmates used. Do they cluster around content (facts, reasoning) or around style and tone, as the Jones and Bergen study found?
2. You judged a roughly 500-million-parameter open model above. The published study used GPT-4, a vastly larger model. Would you expect a smaller model to be easier or harder to detect than GPT-4, and did your own judgement of it support that?
3. The book asks whether, when the AI's reply moved you or made you laugh, you attributed understanding to it, even momentarily. Look back at your own reactions to the model's answers above -- did any of them produce that feeling, and what happened when you reminded yourself it was a 500-million-parameter model with no life to draw on?
4. ELIZA, a very simple 1960s program with none of a modern language model's machinery, still fooled about a fifth of judges. What does that fifth suggest about the baseline rate at which humans are willing to attribute a mind to a machine, regardless of how sophisticated it is?

## Challenge

Run the reverse Turing test properly: ask a friend to hold two blind text conversations, one with you and
one with the small open model above (relayed through you, so they can't tell from response speed), and
have them guess which is which without knowing the true split in advance.

In [ ]:
# Example data: replace with your friend's actual guesses across several blind rounds.
rounds = pd.DataFrame([
    {"round": 1, "actually": "human",   "friend_guess": "human"},    # example data
    {"round": 2, "actually": "chatbot", "friend_guess": "chatbot"},  # example data
    {"round": 3, "actually": "chatbot", "friend_guess": "human"},    # example data
    {"round": 4, "actually": "human",   "friend_guess": "chatbot"},  # example data
])
accuracy = (rounds["actually"] == rounds["friend_guess"]).mean()
print(f"Your friend's accuracy: {accuracy:.0%} across {len(rounds)} rounds")
print("Four rounds is not enough to estimate this precisely -- run more rounds, and note which cues your")
print("friend reports using, to build a slightly larger version of the class_data table above.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-79.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")